In [105]:
from itertools import product
import pandas as pd

class DoorKeyBallEnvironment:
    """
    Ambiente RL discreto para el problema:
    llave -> bola -> puerta -> salida

    Estado:
        s = (R, C, KP, BP, DO)

    Donde:
        R  = fila del agente
        C  = columna del agente
        KP = llave recogida
        BP = bola recogida
        DO = puerta abierta
    """

    def __init__(
        self,
        board=((1, 4), (1, 9)),
        walls=((1, 5), (2, 5), (3, 5)),
        door=(4, 5),
        key=(2, 4),
        ball=(4, 4),
        exit=(1, 7),
        start_state=(3, 1, 0, 0, 0),
        max_steps=50,
    ):
        # Propiedades espaciales
        self.board = board
        (self.min_row, self.max_row), (self.min_col, self.max_col) = board

        self.walls = set(walls)
        self.door = door
        self.key = key
        self.ball = ball
        self.exit = exit
        self.start_state = start_state

        # Acciones
        self.actions = [
            "UP",
            "DOWN",
            "RIGHT",
            "LEFT",
            "PICK_OBJECT",
            "OPEN_DOOR",
        ]

        # Recompensas
        # Penalización por moverse en la Right Room (RR, C > 5)
        # Valor por defecto para movimientos válidos en RR
        self.reward_valid_move = -0.01
        # Penalización mayor para moverse en la Left Room (LR, C < 5)
        self.reward_move_lr = -0.02
        self.reward_invalid_move = -0.1
        self.reward_invalid_open = -1
        self.reward_pick_key = 3
        self.reward_pick_ball = 2
        self.reward_open_door = 5
        self.reward_cross_to_r2 = 1
        self.reward_exit = 15
        self.reward_terminal = 0

        # Control de episodios por longitud
        self.max_steps = max_steps
        self._step_count = 0

    # -----------------------------
    # Operaciones de validación
    # -----------------------------

    def in_board(self, row, col):
        return (
            self.min_row <= row <= self.max_row
            and self.min_col <= col <= self.max_col
        )

    def is_wall(self, row, col):
        return (row, col) in self.walls

    def is_door(self, row, col):
        return (row, col) == self.door

    def is_terminal(self, state):
        row, col, kp, bp, door_open = state
        return (row, col) == self.exit

    def is_blocked(self, row, col, door_open):
        if self.is_wall(row, col):
            return True

        if self.is_door(row, col) and door_open == 0:
            return True

        return False

    def is_valid_agent_position(self, state):
        row, col, kp, bp, door_open = state

        if not self.in_board(row, col):
            return False

        if self.is_wall(row, col):
            return False

        if self.is_door(row, col) and door_open == 0:
            return False

        return True

    # -----------------------------
    # Operaciones de transición
    # -----------------------------

    def move_transition(self, state, new_row, new_col):
        row, col, kp, bp, door_open = state

        if not self.in_board(new_row, new_col):
            return state, self.reward_invalid_move, False

        if self.is_blocked(new_row, new_col, door_open):
            return state, self.reward_invalid_move, False

        next_state = (new_row, new_col, kp, bp, door_open)

        if (row, col) == self.door and (new_row, new_col) == (4, 6):
            return next_state, self.reward_cross_to_r2, False

        # Penalización por moverse depende de la habitación (columna)
        if new_col < 5:
            move_reward = self.reward_move_lr
        else:
            move_reward = self.reward_valid_move

        return next_state, move_reward, False

    def step(self, state, action) -> tuple:
        row, col, kp, bp, door_open = state

        # Si el estado ya es terminal, no ejecutar más acciones
        if self.is_terminal(state):
            return state, self.reward_terminal, True

        # Incrementar contador de pasos solo cuando se avanza en un episodio real
        self._step_count += 1

        if action == "UP":
            next_state, reward, done = self.move_transition(
                state, row - 1, col
            )

        elif action == "DOWN":
            next_state, reward, done = self.move_transition(
                state, row + 1, col
            )

        elif action == "RIGHT":
            next_state, reward, done = self.move_transition(
                state, row, col + 1
            )

        elif action == "LEFT":
            next_state, reward, done = self.move_transition(
                state, row, col - 1
            )

        elif action == "PICK_OBJECT":
            next_state, reward, done = self.pick_object(state)

        elif action == "OPEN_DOOR":
            next_state, reward, done = self.open_door(state)

        else:
            raise ValueError(f"Acción no reconocida: {action}")

        # Si llegamos a estado terminal por movimiento
        if self.is_terminal(next_state):
            return next_state, self.reward_exit, True

        # Si alcanzamos el número máximo de pasos, terminar episodio
        if (self.max_steps is not None) and (self._step_count >= self.max_steps):
            return next_state, reward, True

        return next_state, reward, done

    def pick_object(self, state):
        row, col, kp, bp, door_open = state

        if (row, col) == self.key and kp == 0:
            next_state = (row, col, 1, bp, door_open)
            return next_state, self.reward_pick_key, False

        if (row, col) == self.ball and bp == 0:
            next_state = (row, col, kp, 1, door_open)
            return next_state, self.reward_pick_ball, False

        return state, self.reward_invalid_move, False

    def pick_key(self, state):
        return self.pick_object(state)

    def pick_ball(self, state):
        return self.pick_object(state)

    def open_door(self, state):
        row, col, kp, bp, door_open = state

        # La puerta se abre desde la celda izquierda: (4,4)
        if (
            (row, col) == self.ball
            and kp == 1
            and bp == 1
            and door_open == 0
        ):
            next_state = (row, col, kp, bp, 1)
            return next_state, self.reward_open_door, False

        return state, self.reward_invalid_open, False

    # -----------------------------
    # Operaciones de generación
    # -----------------------------

    def generate_states(self):
        states = []

        for row, col, kp, bp, door_open in product(
            range(self.min_row, self.max_row + 1),
            range(self.min_col, self.max_col + 1),
            [0, 1],
            [0, 1],
            [0, 1],
        ):
            state = (row, col, kp, bp, door_open)

            if self.is_valid_agent_position(state):
                states.append(state)

        return states

    def generate_transition_table(self):
        rows = []

        for state in self.generate_states():
            for action in self.actions:
                # Evitar efectos secundarios en el contador de pasos
                old_count = getattr(self, "_step_count", 0)
                next_state, reward, done = self.step(state, action)
                self._step_count = old_count

                rows.append({
                    "S": state,
                    "A": action,
                    "NS": next_state,
                    "R": reward,
                    "Terminal": done
                })

        return pd.DataFrame(rows)

    def save_transition_table(self, path="transiciones_mdp.csv"):
        df = self.generate_transition_table()
        df.to_csv(path, index=False)
        return df

    def filter_transitions(self, state=None, action=None, terminal=None):
        df = self.generate_transition_table()

        if state is not None:
            df = df[df["S"] == state]

        if action is not None:
            df = df[df["A"] == action]

        if terminal is not None:
            df = df[df["Terminal"] == terminal]

        return df.reset_index(drop=True)

    # -----------------------------
    # Operaciones auxiliares
    # -----------------------------

    def reset(self):
        self._step_count = 0
        return self.start_state

    def describe(self):
        return {
            "board": self.board,
            "walls": self.walls,
            "door": self.door,
            "key": self.key,
            "ball": self.ball,
            "exit": self.exit,
            "start_state": self.start_state,
            "actions": self.actions,
        }

    def get_state_space_definition(self):
        """
        Define el conjunto completo de estados del ambiente.
        Incluye descripción explícita, tipo, rango de valores y características.
        """
        states = self.generate_states()
        
        definition = {
            "description": "Espacio de estados discreto para el problema DoorKeyBall",
            "format": "s = (F, C, KC, BC, DO)",
            "components": {
                "F": {
                    "name": "Fila del agente",
                    "type": "int",
                    "range": (self.min_row, self.max_row),
                    "description": "Posición vertical del agente en el tablero",
                },
                "C": {
                    "name": "Columna del agente",
                    "type": "int",
                    "range": (self.min_col, self.max_col),
                    "description": "Posición horizontal del agente en el tablero",
                },
                "KC": {
                    "name": "Tiene llave",
                    "type": "binary (0 o 1)",
                    "range": (0, 1),
                    "description": "0=sin llave, 1=con llave",
                },
                "BC": {
                    "name": "Tiene bola",
                    "type": "binary (0 o 1)",
                    "range": (0, 1),
                    "description": "0=sin bola, 1=bola recogida",
                },
                "DO": {
                    "name": "Puerta abierta",
                    "type": "binary (0 o 1)",
                    "range": (0, 1),
                    "description": "0=puerta cerrada, 1=puerta abierta",
                },
            },
            "constraints": {
                "wall_positions": list(self.walls),
                "door_position": self.door,
                "key_position": self.key,
                "ball_position": self.ball,
                "exit_position": self.exit,
                "blocked_cells": "Las celdas con muros no son accesibles, ni la puerta cerrada",
            },
            "cardinality": {
                "theoretical_maximum": (self.max_row - self.min_row + 1) * (self.max_col - self.min_col + 1) * 2 * 2 * 2,
                "valid_states": len(states),
                "invalid_states": ((self.max_row - self.min_row + 1) * (self.max_col - self.min_col + 1) * 2 * 2 * 2) - len(states),
            },
            "start_state": self.start_state,
            "terminal_states": f"Cualquier estado donde (F, C) == {self.exit}",
            "state_examples": states[:5],
        }
        
        return definition

    def get_action_space_definition(self):
        """
        Define explícitamente todas las acciones del agente.
        Para cada acción especifica los estados en los cuales es aplicable.
        """
        definition = {
            "description": "Conjunto de acciones disponibles para el agente",
            "total_actions": len(self.actions),
            "actions": {
                "UP": {
                    "name": "Mover arriba",
                    "type": "movimiento",
                    "description": "El agente se mueve hacia arriba (fila - 1)",
                    "applicability": {
                        "condition": "Siempre aplicable",
                        "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada",
                        "reward_if_valid": self.reward_valid_move,
                        "reward_if_invalid": self.reward_invalid_move,
                    }
                },
                "DOWN": {
                    "name": "Mover abajo",
                    "type": "movimiento",
                    "description": "El agente se mueve hacia abajo (fila + 1)",
                    "applicability": {
                        "condition": "Siempre aplicable",
                        "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada",
                        "reward_if_valid": self.reward_valid_move,
                        "reward_if_invalid": self.reward_invalid_move,
                    }
                },
                "RIGHT": {
                    "name": "Mover a la derecha",
                    "type": "movimiento",
                    "description": "El agente se mueve hacia la derecha (columna + 1)",
                    "applicability": {
                        "condition": "Siempre aplicable",
                        "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada",
                    }
                },
                "LEFT": {
                    "name": "Mover a la izquierda",
                    "type": "movimiento",
                    "description": "El agente se mueve hacia la izquierda (columna - 1)",
                    "applicability": {
                        "condition": "Siempre aplicable",
                        "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada",
                    }
                },
                "PICK_OBJECT": {
                    "name": "Recoger objeto",
                    "type": "objeto",
                    "description": "El agente recoge la llave o la bola según la celda",
                    "applicability": {
                        "condition": "Aplicable solo bajo condiciones específicas",
                        "valid_when": f"Agente está en {self.key} con KP=0 O en {self.ball} con BP=0",
                        "invalid_otherwise": "El agente no está sobre un objeto pendiente de recoger",
                        "reward_if_valid": "3 (llave) o 2 (bola)",
                        "reward_if_invalid": self.reward_invalid_move,
                        "effect": "Cambia KC o BC de 0 a 1 según el objeto",
                    }
                },
                "OPEN_DOOR": {
                    "name": "Abrir puerta",
                    "type": "objeto",
                    "description": "El agente abre la puerta cerrada",
                    "applicability": {
                        "condition": "Aplicable solo bajo condiciones específicas",
                        "valid_when": f"Agente está en {self.ball} Y tiene llave recogida (KP=1) Y tiene bola recogida (BP=1) Y puerta cerrada (DO=0)",
                        "invalid_otherwise": "No cumple todas las condiciones anteriores",
                        "reward_if_valid": self.reward_open_door,
                        "reward_if_invalid": self.reward_invalid_open,
                        "effect": "Cambia DO de 0 a 1",
                    }
                }
            },
            "action_summary": {
                "movement_actions": ["UP", "DOWN", "RIGHT", "LEFT"],
                "object_interaction_actions": ["PICK_OBJECT", "OPEN_DOOR"],
                "always_applicable": ["UP", "DOWN", "RIGHT", "LEFT"],
                "conditionally_applicable": ["PICK_OBJECT", "OPEN_DOOR"],
            },
            "state_transitions": {
                "key_locations": {
                    "start": self.key,
                    "interaction": "PICK_OBJECT en posición de la llave"
                },
                "ball_locations": {
                    "start": self.ball,
                    "interaction": "PICK_OBJECT en posición de la bola"
                },
                "door_location": {
                    "position": self.door,
                    "interaction": "OPEN_DOOR abre la puerta si se cumplen todas las condiciones"
                },
                "exit_location": {
                    "position": self.exit,
                    "interaction": "Cualquier movimiento a esta posición termina el episodio"
                }
            },
        }
        
        return definition

    def get_reward_function_definition(self):
        """
        Define la función de recompensa del ambiente.
        Especifica exactamente cada una de las situaciones (parejas estado-acción)
        que ameritan una recompensa con su valor numérico.
        """
        definition = {
            "description": "Función de recompensa que guía el aprendizaje del agente",
            "total_reward_types": 8,
            "reward_values": {
                "reward_valid_move": {
                    "value": self.reward_valid_move,
                    "description": "Movimiento válido",
                    "situations": [
                        "Acción: UP/DOWN/RIGHT/LEFT",
                        "Condición: Nueva posición está dentro del tablero, no es muro, no es puerta cerrada",
                        "Efecto: El agente se desplaza exitosamente",
                    ],
                    "when_applicable": "Cada movimiento exitoso",
                    "motivation": "Penalización pequeña para desincentivar acciones innecesarias",
                },
                "reward_invalid_move": {
                    "value": self.reward_invalid_move,
                    "description": "Movimiento inválido",
                    "situations": [
                        "Acción: UP/DOWN/RIGHT/LEFT",
                        "Condición: Nueva posición fuera del tablero, es un muro, o es puerta cerrada",
                        "Efecto: El agente permanece en su posición actual",
                        "Acción: PICK_OBJECT/OPEN_DOOR",
                        "Condición: Las condiciones específicas no se cumplen",
                        "Efecto: La acción falla sin cambiar el estado",
                    ],
                    "when_applicable": "Cuando se intenta una acción inválida",
                    "motivation": "Penalización mayor para desincentivar acciones inválidas",
                },
                "reward_pick_key": {
                    "value": self.reward_pick_key,
                    "description": "Recoger la llave",
                    "situations": [
                        f"Acción: PICK_OBJECT",
                        f"Condición: Agente en {self.key} Y no tiene llave recogida (KP=0)",
                        "Efecto: KP cambia de 0 a 1",
                    ],
                    "when_applicable": "Cuando el agente recoge la llave",
                    "motivation": "Recompensa positiva para incentivar completar objetivos intermedios",
                },
                "reward_pick_ball": {
                    "value": self.reward_pick_ball,
                    "description": "Recoger la bola",
                    "situations": [
                        f"Acción: PICK_OBJECT",
                        f"Condición: Agente en {self.ball} Y no tiene bola recogida (BP=0)",
                        "Efecto: BP cambia de 0 a 1",
                    ],
                    "when_applicable": "Cuando el agente recoge la bola",
                    "motivation": "Recompensa positiva para incentivar completar objetivos intermedios",
                },
                "reward_open_door": {
                    "value": self.reward_open_door,
                    "description": "Abrir la puerta",
                    "situations": [
                        f"Acción: OPEN_DOOR",
                        f"Condición: Agente en {self.ball} Y KP=1 Y BP=1 Y DO=0",
                        "Efecto: DO cambia de 0 a 1",
                    ],
                    "when_applicable": "Cuando el agente abre la puerta",
                    "motivation": "Recompensa significativa para hito crítico de progreso",
                },
                "reward_invalid_open": {
                    "value": self.reward_invalid_open,
                    "description": "Intento fallido de abrir puerta",
                    "situations": [
                        "Acción: OPEN_DOOR",
                        f"Condición: NO (Agente en {self.ball} Y KP=1 Y BP=1 Y DO=0)",
                        "Efecto: El estado no cambia",
                    ],
                    "when_applicable": "Cuando se intenta abrir la puerta sin cumplir condiciones",
                    "motivation": "Penalización muy fuerte para desincentivar intentos sin completar requisitos",
                },
                "reward_cross_to_r2": {
                    "value": self.reward_cross_to_r2,
                    "description": "Cruzar de región 1 a región 2",
                    "situations": [
                        "Acción: RIGHT",
                        f"Condición: Agente en {self.door} Y se mueve a (4, 6) con puerta abierta (DO=1)",
                        "Efecto: Agente cruza la puerta",
                    ],
                    "when_applicable": "Cuando el agente cruza la puerta abierta",
                    "motivation": "Recompensa moderada por progreso en las regiones del mapa",
                },
                "reward_exit": {
                    "value": self.reward_exit,
                    "description": "Alcanzar la salida",
                    "situations": [
                        "Acción: Cualquier acción",
                        f"Condición: Nueva posición es {self.exit}",
                        "Efecto: El episodio termina (terminal=True)",
                    ],
                    "when_applicable": "Cuando el agente llega a la salida",
                    "motivation": "Recompensa máxima por completar el objetivo final",
                },
                "reward_terminal": {
                    "value": self.reward_terminal,
                    "description": "Estado terminal",
                    "situations": [
                        "Condición: El agente está en la salida y ejecuta cualquier acción",
                        "Efecto: El episodio ya ha terminado, no hay cambio de estado",
                    ],
                    "when_applicable": "En estados terminales",
                    "motivation": "Sin recompensa adicional cuando el episodio ya terminó",
                }
            },
            "reward_structure": {
                "type": "Sparse rewards with dense intermediate rewards",
                "scale": "[-1.0, 15.0]",
                "negative_rewards": [
                    self.reward_valid_move,
                    self.reward_invalid_move,
                    self.reward_invalid_open
                ],
                "positive_rewards": [
                    self.reward_pick_key,
                    self.reward_pick_ball,
                    self.reward_open_door,
                    self.reward_cross_to_r2,
                    self.reward_exit
                ]
            },
            "cumulative_reward_example": {
                "description": "Ejemplo de recompensa acumulativa para resolver el problema",
                "scenario": "Agente sigue camino óptimo: inicio → llave → bola → puerta → salida",
                "path": [
                    "(3,1,0,0,0) → (2,4,0,0,0) [PICK_OBJECT] = +3",
                    "(2,4,1,0,0) → (4,4,1,0,0) [PICK_OBJECT] = +2",
                    "(4,4,1,1,0) → (4,5,1,1,1) [OPEN_DOOR] = +5",
                    "(4,5,1,1,1) → (4,6,1,1,1) [RIGHT] = +1",
                    "(4,6,1,1,1) → (1,7,1,1,1) [movimientos] = múltiples movimientos con -0.01 cada uno",
                    "Llegada a salida (1,7) = +15",
                ],
                "total_optimal": "Mínimo ~10-11 pasos = 3 + 2 + 5 + 1 + 15 + (movimientos) = ~26 - (pasos_extra * 0.01)"
            }
        }
        
        return definition


In [106]:
# ======================================================================
# ACTUALIZACIÓN DE NOTACIÓN DE ESTADOS Y RECOMPENSAS
# ======================================================================

def _state_str(state):
    r, c, kp, bp, do = state
    return f"(R={r}, C={c}, KP={kp}, BP={bp}, DO={do})"


def _full_state_str(row, col, kp='*', bp='*', do='*'):
    return f"(R={row}, C={col}, KP={kp}, BP={bp}, DO={do})"


def get_state_space_definition(self):
    """Define el conjunto completo de estados con la notación R/C/KP/BP/DO."""
    states = self.generate_states()
    return {
        "description": "Espacio de estados discreto para el problema DoorKeyBall",
        "format": "s = (R, C, KP, BP, DO)",
        "components": {
            "R": {"name": "Fila del agente", "type": "int", "range": (self.min_row, self.max_row), "description": "Posición vertical del agente en el tablero"},
            "C": {"name": "Columna del agente", "type": "int", "range": (self.min_col, self.max_col), "description": "Posición horizontal del agente en el tablero"},
            "KP": {"name": "Llave recogida", "type": "binario (0 o 1)", "range": (0, 1), "description": "0=sin recoger la llave, 1=llave recogida"},
            "BP": {"name": "Bola recogida", "type": "binario (0 o 1)", "range": (0, 1), "description": "0=sin recoger la bola, 1=bola recogida"},
            "DO": {"name": "Puerta abierta", "type": "binario (0 o 1)", "range": (0, 1), "description": "0=puerta cerrada, 1=puerta abierta"},
        },
        "constraints": {
            "wall_positions": list(self.walls),
            "door_position": self.door,
            "key_position": self.key,
            "ball_position": self.ball,
            "exit_position": self.exit,
            "blocked_cells": "Las celdas con muros no son accesibles, ni la puerta cerrada",
        },
        "cardinality": {
            "theoretical_maximum": (self.max_row - self.min_row + 1) * (self.max_col - self.min_col + 1) * 2 * 2 * 2,
            "valid_states": len(states),
            "invalid_states": ((self.max_row - self.min_row + 1) * (self.max_col - self.min_col + 1) * 2 * 2 * 2) - len(states),
        },
        "start_state": _state_str(self.start_state),
        "terminal_states": _full_state_str(self.exit[0], self.exit[1], "*", "*", "*"),
        "state_examples": [_state_str(state) for state in states[:5]],
    }


def get_action_space_definition(self):
    """Define las acciones con la nueva notación."""
    move_reward_desc = "Depende: -0.02 si C<5 (LR), -0.01 si C>=5 (RR)"
    return {
        "description": "Conjunto de acciones disponibles para el agente",
        "total_actions": len(self.actions),
        "actions": {
            "UP": {"name": "Mover arriba", "type": "movimiento", "description": "El agente se mueve hacia arriba (fila - 1)", "applicability": {"condition": "Siempre aplicable", "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada", "reward_if_valid": move_reward_desc, "reward_if_invalid": self.reward_invalid_move}},
            "DOWN": {"name": "Mover abajo", "type": "movimiento", "description": "El agente se mueve hacia abajo (fila + 1)", "applicability": {"condition": "Siempre aplicable", "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada", "reward_if_valid": move_reward_desc, "reward_if_invalid": self.reward_invalid_move}},
            "RIGHT": {"name": "Mover a la derecha", "type": "movimiento", "description": "El agente se mueve hacia la derecha (columna + 1)", "applicability": {"condition": "Siempre aplicable", "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada", "reward_if_valid": move_reward_desc, "reward_if_invalid": self.reward_invalid_move, "special_case": f"Si el agente cruza de {self.door} a (4, 6), recibe reward_cross_to_r2={self.reward_cross_to_r2}"}},
            "LEFT": {"name": "Mover a la izquierda", "type": "movimiento", "description": "El agente se mueve hacia la izquierda (columna - 1)", "applicability": {"condition": "Siempre aplicable", "valid_when": "La nueva posición está dentro del tablero, no es un muro y no es una puerta cerrada", "reward_if_valid": move_reward_desc, "reward_if_invalid": self.reward_invalid_move}},
            "PICK_OBJECT": {"name": "Recoger objeto", "type": "objeto", "description": "El agente recoge la llave o la bola según la celda", "applicability": {"condition": "Aplicable solo bajo condiciones específicas", "valid_when": f"Agente está en {self.key} con KP=0 O en {self.ball} con BP=0", "invalid_otherwise": "El agente no está sobre un objeto pendiente de recoger", "reward_if_valid": "3 (llave) o 2 (bola)", "reward_if_invalid": self.reward_invalid_move, "effect": "Cambia KP o BP de 0 a 1 según el objeto"}},
            "OPEN_DOOR": {"name": "Abrir puerta", "type": "objeto", "description": "El agente abre la puerta cerrada", "applicability": {"condition": "Aplicable solo bajo condiciones específicas", "valid_when": f"Agente está en {self.ball} Y tiene llave recogida (KP=1) Y tiene bola recogida (BP=1) Y puerta cerrada (DO=0)", "invalid_otherwise": "No cumple todas las condiciones anteriores", "reward_if_valid": self.reward_open_door, "reward_if_invalid": self.reward_invalid_open, "effect": "Cambia DO de 0 a 1"}},
        },
        "action_summary": {
            "movement_actions": ["UP", "DOWN", "RIGHT", "LEFT"],
            "object_interaction_actions": ["PICK_OBJECT", "OPEN_DOOR"],
            "always_applicable": ["UP", "DOWN", "RIGHT", "LEFT"],
            "conditionally_applicable": ["PICK_OBJECT", "OPEN_DOOR"],
        },
        "state_transitions": {
            "key_locations": {"start": self.key, "interaction": "PICK_OBJECT en posición de la llave"},
            "ball_locations": {"start": self.ball, "interaction": "PICK_OBJECT en posición de la bola"},
            "door_location": {"position": self.door, "interaction": "OPEN_DOOR abre la puerta si se cumplen todas las condiciones"},
            "exit_location": {"position": self.exit, "interaction": "Cualquier movimiento a esta posición termina el episodio"},
        },
    }


def get_reward_function_definition(self):
    """Define la función de recompensa con la nueva notación de estado."""
    return {
        "description": "Función de recompensa que guía el aprendizaje del agente",
        "total_reward_types": 8,
        "reward_values": {
            "reward_valid_move": {"value": self.reward_valid_move, "description": "Movimiento válido", "situations": ["Acción: UP/DOWN/RIGHT/LEFT", "Condición: Nueva posición está dentro del tablero, no es muro, no es puerta cerrada", "Efecto: El agente se desplaza exitosamente"], "when_applicable": "Cada movimiento exitoso", "motivation": "Penalización pequeña para desincentivar acciones innecesarias"},
            "reward_invalid_move": {"value": self.reward_invalid_move, "description": "Movimiento inválido", "situations": ["Acción: UP/DOWN/RIGHT/LEFT", "Condición: Nueva posición fuera del tablero, es un muro, o es puerta cerrada", "Efecto: El agente permanece en su posición actual", "Acción: PICK_OBJECT/OPEN_DOOR", "Condición: Las condiciones específicas no se cumplen", "Efecto: La acción falla sin cambiar el estado"], "when_applicable": "Cuando se intenta una acción inválida", "motivation": "Penalización mayor para desincentivar acciones inválidas"},
            "reward_pick_key": {"value": self.reward_pick_key, "description": "Recoger la llave", "situations": ["Acción: PICK_OBJECT", f"Condición: Agente en {self.key} con KP=0, BP=0, DO=0", "Efecto: KP cambia de 0 a 1"], "when_applicable": "Cuando el agente recoge la llave", "motivation": "Recompensa positiva para incentivar completar objetivos intermedios"},
            "reward_pick_ball": {"value": self.reward_pick_ball, "description": "Recoger la bola", "situations": ["Acción: PICK_OBJECT", f"Condición: Agente en {self.ball} con KP=1, BP=0, DO=0", "Efecto: BP cambia de 0 a 1"], "when_applicable": "Cuando el agente recoge la bola", "motivation": "Recompensa positiva para incentivar completar objetivos intermedios"},
            "reward_open_door": {"value": self.reward_open_door, "description": "Abrir la puerta", "situations": ["Acción: OPEN_DOOR", f"Condición: Agente en {self.ball} con KP=1, BP=1, DO=0", "Efecto: DO cambia de 0 a 1"], "when_applicable": "Cuando el agente abre la puerta", "motivation": "Recompensa significativa para hito crítico de progreso"},
            "reward_invalid_open": {"value": self.reward_invalid_open, "description": "Intento fallido de abrir puerta", "situations": ["Acción: OPEN_DOOR", f"Condición: NO (Agente en {self.ball} con KP=1, BP=1, DO=0)", "Efecto: El estado no cambia"], "when_applicable": "Cuando se intenta abrir la puerta sin cumplir condiciones", "motivation": "Penalización muy fuerte para desincentivar intentos sin completar requisitos"},
            "reward_cross_to_r2": {"value": self.reward_cross_to_r2, "description": "Cruzar de región 1 a región 2", "situations": ["Acción: RIGHT", f"Condición: Agente en {self.door} con KP=1, BP=1, DO=1 y se mueve a (R=4, C=6, KP=1, BP=1, DO=1)", "Efecto: Agente cruza la puerta"], "when_applicable": "Cuando el agente cruza la puerta abierta", "motivation": "Recompensa moderada por progreso en las regiones del mapa"},
            "reward_exit": {"value": self.reward_exit, "description": "Alcanzar la salida", "situations": ["Acción: Cualquier acción", f"Condición: Nueva posición es (R={self.exit[0]}, C={self.exit[1]}, KP=*, BP=*, DO=*)", "Efecto: El episodio termina (terminal=True)"], "when_applicable": "Cuando el agente llega a la salida", "motivation": "Recompensa máxima por completar el objetivo final"},
            "reward_terminal": {"value": self.reward_terminal, "description": "Estado terminal", "situations": [f"Condición: El agente está en (R={self.exit[0]}, C={self.exit[1]}, KP=*, BP=*, DO=*) y ejecuta cualquier acción", "Efecto: El episodio ya ha terminado, no hay cambio de estado"], "when_applicable": "En estados terminales", "motivation": "Sin recompensa adicional cuando el episodio ya terminó"},
        },
        "reward_structure": {"type": "Sparse rewards with dense intermediate rewards", "scale": "[-1.0, 15.0]", "negative_rewards": [self.reward_valid_move, self.reward_invalid_move, self.reward_invalid_open], "positive_rewards": [self.reward_pick_key, self.reward_pick_ball, self.reward_open_door, self.reward_cross_to_r2, self.reward_exit]},
        "state_action_reward_table": [
            {"Estado": _full_state_str(self.start_state[0], self.start_state[1], self.start_state[2], self.start_state[3], self.start_state[4]), "Acción": "ANY", "Recompensa": self.reward_terminal, "Descripción": "Estado inicial; si ya es terminal, no hay recompensa adicional"},
            {"Estado": _full_state_str(self.key[0], self.key[1], 0, 0, 0), "Acción": "PICK_OBJECT", "Recompensa": self.reward_pick_key, "Descripción": "Recoger la llave"},
            {"Estado": _full_state_str(self.ball[0], self.ball[1], 1, 0, 0), "Acción": "PICK_OBJECT", "Recompensa": self.reward_pick_ball, "Descripción": "Recoger la bola"},
            {"Estado": _full_state_str(self.ball[0], self.ball[1], 1, 1, 0), "Acción": "OPEN_DOOR", "Recompensa": self.reward_open_door, "Descripción": "Abrir la puerta"},
            {"Estado": _full_state_str(self.door[0], self.door[1], 1, 1, 1), "Acción": "RIGHT", "Recompensa": self.reward_cross_to_r2, "Descripción": "Cruzar de la región 1 a la región 2"},
            {"Estado": "Cualquier estado válido", "Acción": "UP/DOWN/RIGHT/LEFT", "Recompensa": self.reward_valid_move, "Descripción": "Movimiento válido"},
            {"Estado": "Cualquier estado inválido para la acción", "Acción": "UP/DOWN/RIGHT/LEFT", "Recompensa": self.reward_invalid_move, "Descripción": "Movimiento inválido"},
            {"Estado": "Estado que no cumple condiciones", "Acción": "OPEN_DOOR", "Recompensa": self.reward_invalid_open, "Descripción": "Intento fallido de abrir la puerta"},
            {"Estado": _full_state_str(self.exit[0], self.exit[1], "*", "*", "*"), "Acción": "ANY", "Recompensa": self.reward_exit, "Descripción": "Alcanzar la salida"},
        ],
        "cumulative_reward_example": {"description": "Ejemplo de recompensa acumulativa para resolver el problema", "scenario": "Agente sigue camino óptimo: inicio → llave → bola → puerta → salida", "path": ["(R=3, C=1, KP=0, BP=0, DO=0) → (R=2, C=4, KP=0, BP=0, DO=0) [PICK_OBJECT] = +3", "(R=2, C=4, KP=1, BP=0, DO=0) → (R=4, C=4, KP=1, BP=0, DO=0) [PICK_OBJECT] = +2", "(R=4, C=4, KP=1, BP=1, DO=0) → (R=4, C=5, KP=1, BP=1, DO=1) [OPEN_DOOR] = +5", "(R=4, C=5, KP=1, BP=1, DO=1) → (R=4, C=6, KP=1, BP=1, DO=1) [RIGHT] = +1", "(R=4, C=6, KP=1, BP=1, DO=1) → (R=1, C=7, KP=1, BP=1, DO=1) [movimientos] = múltiples movimientos con -0.01 cada uno", "Llegada a salida (R=1, C=7, KP=1, BP=1, DO=1) = +15"], "total_optimal": "Mínimo ~10-11 pasos = 3 + 2 + 5 + 1 + 15 + (movimientos) = ~26 - (pasos_extra * 0.01)"},
    }


DoorKeyBallEnvironment.get_state_space_definition = get_state_space_definition
DoorKeyBallEnvironment.get_action_space_definition = get_action_space_definition
DoorKeyBallEnvironment.get_reward_function_definition = get_reward_function_definition

In [107]:
# ======================================================================
# EXTENSIÓN: tabla explícita Estado - Acción - Recompensa (actualizada para LR/RR)
# ======================================================================

def get_reward_function_definition(self):
    """
    Define la función de recompensa del ambiente.
    Especifica exactamente cada una de las situaciones (parejas estado-acción)
    que ameritan una recompensa con su valor numérico.
    """
    definition = {
        "description": "Función de recompensa que guía el aprendizaje del agente",
        "total_reward_types": 8,
        "reward_values": {
            "reward_valid_move": {
                "value": f"LR: {self.reward_move_lr}, RR: {self.reward_valid_move}",
                "description": "Movimiento válido (penalización depende de la columna)",
                "situations": [
                    "Acción: UP/DOWN/RIGHT/LEFT",
                    "Condición: Nueva posición está dentro del tablero, no es muro, no es puerta cerrada",
                    "Efecto: El agente se desplaza exitosamente",
                ],
                "when_applicable": "Cada movimiento exitoso (LR: C<5; RR: C>=5)",
                "motivation": "Penalización pequeña para desincentivar acciones innecesarias; LR tiene penalización mayor",
            },
            "reward_invalid_move": {
                "value": self.reward_invalid_move,
                "description": "Movimiento inválido",
                "situations": [
                    "Acción: UP/DOWN/RIGHT/LEFT",
                    "Condición: Nueva posición fuera del tablero, es un muro, o es puerta cerrada",
                    "Efecto: El agente permanece en su posición actual",
                    "Acción: PICK_OBJECT/OPEN_DOOR",
                    "Condición: Las condiciones específicas no se cumplen",
                    "Efecto: La acción falla sin cambiar el estado",
                ],
                "when_applicable": "Cuando se intenta una acción inválida",
                "motivation": "Penalización mayor para desincentivar acciones inválidas",
            },
            "reward_pick_key": {
                "value": self.reward_pick_key,
                "description": "Recoger la llave",
                "situations": [
                    "Acción: PICK_OBJECT",
                    f"Condición: Agente en (R={self.key[0]}, C={self.key[1]}, KP=0, BP=0, DO=0)",
                    "Efecto: KP cambia de 0 a 1",
                ],
                "when_applicable": "Cuando el agente recoge la llave",
                "motivation": "Recompensa positiva para incentivar completar objetivos intermedios",
            },
            "reward_pick_ball": {
                "value": self.reward_pick_ball,
                "description": "Recoger la bola",
                "situations": [
                    "Acción: PICK_OBJECT",
                    f"Condición: Agente en (R={self.ball[0]}, C={self.ball[1]}, KP=1, BP=0, DO=0)",
                    "Efecto: BP cambia de 0 a 1",
                ],
                "when_applicable": "Cuando el agente recoge la bola",
                "motivation": "Recompensa positiva para incentivar completar objetivos intermedios",
            },
            "reward_open_door": {
                "value": self.reward_open_door,
                "description": "Abrir la puerta",
                "situations": [
                    "Acción: OPEN_DOOR",
                    f"Condición: Agente en (R={self.ball[0]}, C={self.ball[1]}, KP=1, BP=1, DO=0)",
                    "Efecto: DO cambia de 0 a 1",
                ],
                "when_applicable": "Cuando el agente abre la puerta",
                "motivation": "Recompensa significativa para hito crítico de progreso",
            },
            "reward_invalid_open": {
                "value": self.reward_invalid_open,
                "description": "Intento fallido de abrir puerta",
                "situations": [
                    "Acción: OPEN_DOOR",
                    f"Condición: NO (Agente en (R={self.ball[0]}, C={self.ball[1]}, KP=1, BP=1, DO=0))",
                    "Efecto: El estado no cambia",
                ],
                "when_applicable": "Cuando se intenta abrir la puerta sin cumplir condiciones",
                "motivation": "Penalización muy fuerte para desincentivar intentos sin completar requisitos",
            },
            "reward_cross_to_r2": {
                "value": self.reward_cross_to_r2,
                "description": "Cruzar de región 1 a región 2",
                "situations": [
                    "Acción: RIGHT",
                    f"Condición: Agente en (R={self.door[0]}, C={self.door[1]}, KP=1, BP=1, DO=1) Y se mueve a (R=4, C=6, KP=1, BP=1, DO=1)",
                    "Efecto: Agente cruza la puerta",
                ],
                "when_applicable": "Cuando el agente cruza la puerta abierta",
                "motivation": "Recompensa moderada por progreso en las regiones del mapa",
            },
            "reward_exit": {
                "value": self.reward_exit,
                "description": "Alcanzar la salida",
                "situations": [
                    "Acción: Cualquier acción",
                    f"Condición: Nueva posición es (R={self.exit[0]}, C={self.exit[1]}, KP=*, BP=*, DO=*)",
                    "Efecto: El episodio termina (terminal=True)",
                ],
                "when_applicable": "Cuando el agente llega a la salida",
                "motivation": "Recompensa máxima por completar el objetivo final",
            },
            "reward_terminal": {
                "value": self.reward_terminal,
                "description": "Estado terminal",
                "situations": [
                    f"Condición: El agente está en (R={self.exit[0]}, C={self.exit[1]}, KP=*, BP=*, DO=*) y ejecuta cualquier acción",
                    "Efecto: El episodio ya ha terminado, no hay cambio de estado",
                ],
                "when_applicable": "En estados terminales",
                "motivation": "Sin recompensa adicional cuando el episodio ya terminó",
            }
        },
        "reward_structure": {
            "type": "Sparse rewards with dense intermediate rewards",
            "scale": "[-1.0, 15.0]",
            "negative_rewards": [
                self.reward_move_lr,
                self.reward_valid_move,
                self.reward_invalid_move,
                self.reward_invalid_open
            ],
            "positive_rewards": [
                self.reward_pick_key,
                self.reward_pick_ball,
                self.reward_open_door,
                self.reward_cross_to_r2,
                self.reward_exit
            ]
        },
        "state_action_reward_table": [
            {
                "Estado": f"(R={self.start_state[0]}, C={self.start_state[1]}, KP={self.start_state[2]}, BP={self.start_state[3]}, DO={self.start_state[4]})",
                "Acción": "ANY",
                "Recompensa": self.reward_terminal,
                "Descripción": "Estado inicial; si ya es terminal, no hay recompensa adicional",
            },
            {
                "Estado": f"(R={self.key[0]}, C={self.key[1]}, KP=0, BP=0, DO=0)",
                "Acción": "PICK_OBJECT",
                "Recompensa": self.reward_pick_key,
                "Descripción": "Recoger la llave",
            },
            {
                "Estado": f"(R={self.ball[0]}, C={self.ball[1]}, KP=1, BP=0, DO=0)",
                "Acción": "PICK_OBJECT",
                "Recompensa": self.reward_pick_ball,
                "Descripción": "Recoger la bola",
            },
            {
                "Estado": f"(R={self.ball[0]}, C={self.ball[1]}, KP=1, BP=1, DO=0)",
                "Acción": "OPEN_DOOR",
                "Recompensa": self.reward_open_door,
                "Descripción": "Abrir la puerta",
            },
            {
                "Estado": f"(R={self.door[0]}, C={self.door[1]}, KP=1, BP=1, DO=1)",
                "Acción": "RIGHT",
                "Recompensa": self.reward_cross_to_r2,
                "Descripción": "Cruzar de la región 1 a la región 2",
            },
            {
                "Estado": "Cualquier estado válido",
                "Acción": "UP/DOWN/RIGHT/LEFT",
                "Recompensa": f"Depende de la columna: {self.reward_move_lr} si C<5 (LR) / {self.reward_valid_move} si C>=5 (RR)",
                "Descripción": "Movimiento válido (LR penalizado más que RR)",
            },
            {
                "Estado": "Cualquier estado inválido para la acción",
                "Acción": "UP/DOWN/RIGHT/LEFT",
                "Recompensa": self.reward_invalid_move,
                "Descripción": "Movimiento inválido",
            },
            {
                "Estado": "Estado que no cumple condiciones",
                "Acción": "OPEN_DOOR",
                "Recompensa": self.reward_invalid_open,
                "Descripción": "Intento fallido de abrir la puerta",
            },
            {
                "Estado": f"(R={self.exit[0]}, C={self.exit[1]}, KP=*, BP=*, DO=*)",
                "Acción": "ANY",
                "Recompensa": self.reward_exit,
                "Descripción": "Alcanzar la salida",
            },
        ],
        "cumulative_reward_example": {
            "description": "Ejemplo de recompensa acumulativa para resolver el problema",
            "scenario": "Agente sigue camino óptimo: inicio → llave → bola → puerta → salida",
            "path": [
                "(R=3, C=1, KP=0, BP=0, DO=0) → (R=2, C=4, KP=0, BP=0, DO=0) [PICK_OBJECT] = +3",
                "(R=2, C=4, KP=1, BP=0, DO=0) → (R=4, C=4, KP=1, BP=0, DO=0) [PICK_OBJECT] = +2",
                "(R=4, C=4, KP=1, BP=1, DO=0) → (R=4, C=5, KP=1, BP=1, DO=1) [OPEN_DOOR] = +5",
                "(R=4, C=5, KP=1, BP=1, DO=1) → (R=4, C=6, KP=1, BP=1, DO=1) [RIGHT] = +1",
                "(R=4, C=6, KP=1, BP=1, DO=1) → (R=1, C=7, KP=1, BP=1, DO=1) [movimientos] = múltiples movimientos con penalización dependiente de la columna (LR=-0.02, RR=-0.01)",
                "Llegada a salida (R=1, C=7, KP=1, BP=1, DO=1) = +15",
            ],
            "total_optimal": "Mínimo ~10-11 pasos = 3 + 2 + 5 + 1 + 15 + (movimientos) = ~26 - (pasos_extra * penalización por movimiento según habitación)"
        }
    }
    
    return definition

DoorKeyBallEnvironment.get_reward_function_definition = get_reward_function_definition


In [108]:
env = DoorKeyBallEnvironment()

In [109]:
# Prueba rápida: penalizaciones LR vs RR
env = DoorKeyBallEnvironment()

# LR: columna < 5 -> penalización esperada -0.02
state_lr = (3, 1, 0, 0, 0)
ns_lr, r_lr, d_lr = env.step(state_lr, "RIGHT")
print(state_lr, "RIGHT ->", ns_lr, "reward", r_lr)

# RR: columna > 5 -> penalización esperada -0.01
state_rr = (3, 7, 0, 0, 1)
ns_rr, r_rr, d_rr = env.step(state_rr, "LEFT")
print(state_rr, "LEFT ->", ns_rr, "reward", r_rr)

# Movimiento cruzando hacia columna 5 (pasillo) - se tratará como RR (C>=5)
state_corr = (3, 6, 0, 0, 1)
ns_corr, r_corr, d_corr = env.step(state_corr, "LEFT")
print(state_corr, "LEFT ->", ns_corr, "reward", r_corr)

(3, 1, 0, 0, 0) RIGHT -> (3, 2, 0, 0, 0) reward -0.02
(3, 7, 0, 0, 1) LEFT -> (3, 6, 0, 0, 1) reward -0.01
(3, 6, 0, 0, 1) LEFT -> (3, 6, 0, 0, 1) reward -0.1


In [110]:
import json
import pandas as pd

# Obtener la definición completa del espacio de estados
state_space_def = env.get_state_space_definition()
 
# Mostrar la definición en formato legible
print(f"\n{'='*80}")
print(f"DEFINICIÓN DEL ESPACIO DE ESTADOS - DoorKeyBall Environment")
print(f"{'='*80}\n")
print(f"Descripción: {state_space_def['description']}")
print(f"Formato: {state_space_def['format']}\n")

# TABLA 1: COMPONENTES DEL ESTADO
print("TABLA 1: COMPONENTES DEL ESTADO")
print("-" * 80)
components_data = []
for component, details in state_space_def['components'].items():
    components_data.append({
        'Componente': component,
        'Nombre': details['name'],
        'Tipo': details['type'],
        'Rango': str(details['range']),
        'Descripción': details['description']
    })
df_components = pd.DataFrame(components_data)
print(df_components.to_string(index=False))

# TABLA 2: RESTRICCIONES
print("\n\nTABLA 2: RESTRICCIONES")
print("-" * 80)
constraints_data = []
for constraint, value in state_space_def['constraints'].items():
    if isinstance(value, list):
        constraints_data.append({'Restricción': constraint, 'Valor': str(value)})
    else:
        constraints_data.append({'Restricción': constraint, 'Valor': str(value)})
df_constraints = pd.DataFrame(constraints_data)
print(df_constraints.to_string(index=False))

# TABLA 3: CARDINALIDAD
print("\n\nTABLA 3: CARDINALIDAD DEL ESPACIO DE ESTADOS")
print("-" * 80)
cardinality_data = []
for key, value in state_space_def['cardinality'].items():
    cardinality_data.append({'Métrica': key, 'Valor': value})
df_cardinality = pd.DataFrame(cardinality_data)
print(df_cardinality.to_string(index=False))

# Información adicional
print(f"\n\nESTADO INICIAL:")
print("-" * 80)
print(f"{state_space_def['start_state']}")

print(f"\n\nESTADOS TERMINALES:")
print("-" * 80)
print(f"{state_space_def['terminal_states']}")

# TABLA 4: EJEMPLOS DE ESTADOS VÁLIDOS
print("\n\nTABLA 4: EJEMPLOS DE ESTADOS VÁLIDOS (primeros 5)")
print("-" * 80)
examples_data = []
for i, state in enumerate(state_space_def['state_examples'], 1):
    examples_data.append({'#': i, 'Estado (R, C, KP, BP, DO)': str(state)})
df_examples = pd.DataFrame(examples_data)
print(df_examples.to_string(index=False))


DEFINICIÓN DEL ESPACIO DE ESTADOS - DoorKeyBall Environment

Descripción: Espacio de estados discreto para el problema DoorKeyBall
Formato: s = (R, C, KP, BP, DO)

TABLA 1: COMPONENTES DEL ESTADO
--------------------------------------------------------------------------------
Componente             Nombre            Tipo  Rango                                  Descripción
         R    Fila del agente             int (1, 4)   Posición vertical del agente en el tablero
         C Columna del agente             int (1, 9) Posición horizontal del agente en el tablero
        KP     Llave recogida binario (0 o 1) (0, 1)     0=sin recoger la llave, 1=llave recogida
        BP      Bola recogida binario (0 o 1) (0, 1)       0=sin recoger la bola, 1=bola recogida
        DO     Puerta abierta binario (0 o 1) (0, 1)           0=puerta cerrada, 1=puerta abierta


TABLA 2: RESTRICCIONES
--------------------------------------------------------------------------------
   Restricción              

In [111]:
# Obtener la definición completa del espacio de acciones
action_space_def = env.get_action_space_definition()

print("\n" + "=" * 100)
print("DEFINICIÓN DEL ESPACIO DE ACCIONES - DoorKeyBall Environment")
print("=" * 100)
print(f"\nDescripción: {action_space_def['description']}")
print(f"Total de acciones: {action_space_def['total_actions']}\n")

# TABLA 1: RESUMEN DE ACCIONES
print("TABLA 1: CLASIFICACIÓN DE ACCIONES")
print("-" * 100)
summary = action_space_def['action_summary']
summary_data = [
    {'Categoría': 'Acciones de movimiento', 'Acciones': str(summary['movement_actions'])},
    {'Categoría': 'Acciones de interacción', 'Acciones': str(summary['object_interaction_actions'])},
    {'Categoría': 'Siempre aplicables', 'Acciones': str(summary['always_applicable'])},
    {'Categoría': 'Condicionalmente aplicables', 'Acciones': str(summary['conditionally_applicable'])},
]
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

# TABLA 2: DETALLES DE ACCIONES
print("\n\nTABLA 2: DETALLES DE CADA ACCIÓN")
print("-" * 100)
actions_data = []
for action_name, action_details in action_space_def['actions'].items():
    applicability = action_details['applicability']
    actions_data.append({
        'Acción': action_name,
        'Nombre': action_details['name'],
        'Tipo': action_details['type'],
        'Condición': applicability.get('condition', 'N/A'),
        'Válida cuando': applicability.get('valid_when', 'N/A')[:60] + '...' if len(applicability.get('valid_when', 'N/A')) > 60 else applicability.get('valid_when', 'N/A'),
    })
df_actions = pd.DataFrame(actions_data)
print(df_actions.to_string(index=False))

# TABLA 3: RECOMPENSAS POR ACCIÓN
print("\n\nTABLA 3: RECOMPENSAS POR ACCIÓN")
print("-" * 100)
rewards_data = []
for action_name, action_details in action_space_def['actions'].items():
    applicability = action_details['applicability']
    rewards_data.append({
        'Acción': action_name,
        'Recompensa (válida)': applicability.get('reward_if_valid', 'N/A'),
        'Recompensa (inválida)': applicability.get('reward_if_invalid', 'N/A'),
    })
df_rewards = pd.DataFrame(rewards_data)
print(df_rewards.to_string(index=False))

# TABLA 4: TRANSICIONES DE ESTADO
print("\n\nTABLA 4: UBICACIONES CLAVE Y SUS INTERACCIONES")
print("-" * 100)
transitions = action_space_def['state_transitions']
transitions_data = []
for location_type, location_info in transitions.items():
    transitions_data.append({
        'Ubicación': location_type,
        'Posición': str(location_info.get('start', location_info.get('position', 'N/A'))),
        'Interacción': location_info.get('interaction', 'N/A'),
    })
df_transitions = pd.DataFrame(transitions_data)
print(df_transitions.to_string(index=False))


DEFINICIÓN DEL ESPACIO DE ACCIONES - DoorKeyBall Environment

Descripción: Conjunto de acciones disponibles para el agente
Total de acciones: 6

TABLA 1: CLASIFICACIÓN DE ACCIONES
----------------------------------------------------------------------------------------------------
                  Categoría                        Acciones
     Acciones de movimiento ['UP', 'DOWN', 'RIGHT', 'LEFT']
    Acciones de interacción    ['PICK_OBJECT', 'OPEN_DOOR']
         Siempre aplicables ['UP', 'DOWN', 'RIGHT', 'LEFT']
Condicionalmente aplicables    ['PICK_OBJECT', 'OPEN_DOOR']


TABLA 2: DETALLES DE CADA ACCIÓN
----------------------------------------------------------------------------------------------------
     Acción               Nombre       Tipo                                   Condición                                                   Válida cuando
         UP         Mover arriba movimiento                           Siempre aplicable La nueva posición está dentro del tablero,

In [112]:
# Demostración de get_reward_function_definition()
print("=" * 120)
print("DEFINICIÓN DE LA FUNCIÓN DE RECOMPENSA - DoorKeyBall Environment")
print("=" * 120)

reward_def = env.get_reward_function_definition()

print(f"\nDescripción: {reward_def['description']}")
print(f"Tipos totales de recompensas: {reward_def['total_reward_types']}\n")

# TABLA 1: RESUMEN DE RECOMPENSAS
print("TABLA 1: RESUMEN DE TIPOS DE RECOMPENSAS")
print("-" * 120)
rewards_summary = []
for reward_type, reward_info in reward_def['reward_values'].items():
    rewards_summary.append({
        'Tipo': reward_type,
        'Valor': reward_info['value'],
        'Descripción': reward_info['description'],
        'Cuándo aplica': reward_info['when_applicable'],
        'Motivación': reward_info['motivation']
    })
df_rewards_summary = pd.DataFrame(rewards_summary)
print(df_rewards_summary.to_string(index=False))

# TABLA 2: ESTRUCTURA DE RECOMPENSAS
print("\n\nTABLA 2: ESTRUCTURA DE RECOMPENSAS")
print("-" * 120)
reward_struct = reward_def['reward_structure']
structure_data = [
    {'Aspecto': 'Tipo de estructura', 'Valor': reward_struct['type']},
    {'Aspecto': 'Escala de valores', 'Valor': reward_struct['scale']},
    {'Aspecto': 'Recompensas negativas (penalizaciones)', 'Valor': str(reward_struct['negative_rewards'])},
    {'Aspecto': 'Recompensas positivas (incentivos)', 'Valor': str(reward_struct['positive_rewards'])},
]
df_structure = pd.DataFrame(structure_data)
print(df_structure.to_string(index=False))

# TABLA 3: SITUACIONES POR TIPO DE RECOMPENSA
print("\n\nTABLA 3: SITUACIONES (PARES ESTADO-ACCIÓN) QUE GENERAN RECOMPENSA")
print("-" * 120)
situations_data = []
for reward_type, reward_info in reward_def['reward_values'].items():
    situations_text = " | ".join(reward_info['situations'])
    situations_data.append({
        'Recompensa': reward_type,
        'Valor': reward_info['value'],
        'Situaciones': situations_text
    })
df_situations = pd.DataFrame(situations_data)
print(df_situations.to_string(index=False))

# TABLA 4: RUTA ÓPTIMA CON RECOMPENSAS ACUMULATIVAS
print("\n\nTABLA 4: EJEMPLO DE RUTA ÓPTIMA (CAMINO ÓPTIMO)")
print("-" * 120)
example = reward_def['cumulative_reward_example']
print(f"Escenario: {example['scenario']}\n")
route_data = []
for i, step in enumerate(example['path'], 1):
    route_data.append({'Paso': i, 'Transición': step})
df_route = pd.DataFrame(route_data)
print(df_route.to_string(index=False))
print(f"\nRecompensa total óptima: {example['total_optimal']}")

# TABLA 5: ESTADO - ACCIÓN - RECOMPENSA
print("\n\nTABLA 5: ESTADO - ACCIÓN - RECOMPENSA")
print("-" * 120)
state_action_data = reward_def['state_action_reward_table']
df_state_action = pd.DataFrame(state_action_data)
print(df_state_action.to_string(index=False))

print("\n" + "=" * 120)
print("RESUMEN FINAL: 8 tipos de recompensas definiendo todas las parejas (estado, acción)")
print("=" * 120)

DEFINICIÓN DE LA FUNCIÓN DE RECOMPENSA - DoorKeyBall Environment

Descripción: Función de recompensa que guía el aprendizaje del agente
Tipos totales de recompensas: 8

TABLA 1: RESUMEN DE TIPOS DE RECOMPENSAS
------------------------------------------------------------------------------------------------------------------------
               Tipo                Valor                                            Descripción                                             Cuándo aplica                                                                                 Motivación
  reward_valid_move LR: -0.02, RR: -0.01 Movimiento válido (penalización depende de la columna)               Cada movimiento exitoso (LR: C<5; RR: C>=5) Penalización pequeña para desincentivar acciones innecesarias; LR tiene penalización mayor
reward_invalid_move                 -0.1                                    Movimiento inválido                     Cuando se intenta una acción inválida                          

In [113]:
import sys
import io

# ============================================================================
# GUARDAR SALIDAS DE CELDAS 3, 4 Y 5 EN ARCHIVOS DE TEXTO
# ============================================================================

# Ruta base para guardar archivos
import os
output_dir = "outputs_definiciones"
os.makedirs(output_dir, exist_ok=True)

# ============================================================================
# GUARDAR CELDA 3: DEFINICIÓN DEL ESPACIO DE ESTADOS
# ============================================================================
output_buffer = io.StringIO()
old_stdout = sys.stdout
sys.stdout = output_buffer

state_space_def = env.get_state_space_definition()
 
print(f"\n{'='*80}")
print(f"DEFINICIÓN DEL ESPACIO DE ESTADOS - DoorKeyBall Environment")
print(f"{'='*80}\n")
print(f"Descripción: {state_space_def['description']}")
print(f"Formato: {state_space_def['format']}\n")

print("TABLA 1: COMPONENTES DEL ESTADO")
print("-" * 80)
components_data = []
for component, details in state_space_def['components'].items():
    components_data.append({
        'Componente': component,
        'Nombre': details['name'],
        'Tipo': details['type'],
        'Rango': str(details['range']),
        'Descripción': details['description']
    })
df_components = pd.DataFrame(components_data)
print(df_components.to_string(index=False))

print("\n\nTABLA 2: RESTRICCIONES")
print("-" * 80)
constraints_data = []
for constraint, value in state_space_def['constraints'].items():
    if isinstance(value, list):
        constraints_data.append({'Restricción': constraint, 'Valor': str(value)})
    else:
        constraints_data.append({'Restricción': constraint, 'Valor': str(value)})
df_constraints = pd.DataFrame(constraints_data)
print(df_constraints.to_string(index=False))

print("\n\nTABLA 3: CARDINALIDAD DEL ESPACIO DE ESTADOS")
print("-" * 80)
cardinality_data = []
for key, value in state_space_def['cardinality'].items():
    cardinality_data.append({'Métrica': key, 'Valor': value})
df_cardinality = pd.DataFrame(cardinality_data)
print(df_cardinality.to_string(index=False))

print(f"\n\nESTADO INICIAL:")
print("-" * 80)
start_state = state_space_def['start_state']
if isinstance(start_state, tuple) and len(start_state) == 5:
    print(f"(R={start_state[0]}, C={start_state[1]}, KP={start_state[2]}, BP={start_state[3]}, DO={start_state[4]})")
else:
    print(f"{start_state}")

print(f"\n\nESTADOS TERMINALES:")
print("-" * 80)
terminal_states = state_space_def['terminal_states']
if isinstance(terminal_states, str) and '(R=' not in terminal_states:
    print(f"Cualquier estado donde (R={env.exit[0]}, C={env.exit[1]}, KP=*, BP=*, DO=*)")
else:
    print(f"{terminal_states}")

print("\n\nTABLA 4: EJEMPLOS DE ESTADOS VÁLIDOS (primeros 5)")
print("-" * 80)
examples_data = []
for i, state in enumerate(state_space_def['state_examples'], 1):
    if isinstance(state, tuple) and len(state) == 5:
        state_text = f"(R={state[0]}, C={state[1]}, KP={state[2]}, BP={state[3]}, DO={state[4]})"
    else:
        state_text = str(state)
    examples_data.append({'#': i, 'Estado (R, C, KP, BP, DO)': state_text})
df_examples = pd.DataFrame(examples_data)
print(df_examples.to_string(index=False))

# Guardar en archivo
sys.stdout = old_stdout
output_text_cell3 = output_buffer.getvalue()
with open(os.path.join(output_dir, "celda_3_espacio_de_estados.txt"), "w", encoding="utf-8") as f:
    f.write(output_text_cell3)
print(f"✓ Salida de celda 3 guardada en: {output_dir}/celda_3_espacio_de_estados.txt")

# ============================================================================
# GUARDAR CELDA 4: DEFINICIÓN DEL ESPACIO DE ACCIONES
# ============================================================================
output_buffer = io.StringIO()
sys.stdout = output_buffer

action_space_def = env.get_action_space_definition()

print("\n" + "=" * 100)
print("DEFINICIÓN DEL ESPACIO DE ACCIONES - DoorKeyBall Environment")
print("=" * 100)
print(f"\nDescripción: {action_space_def['description']}")
print(f"Total de acciones: {action_space_def['total_actions']}\n")

print("TABLA 1: CLASIFICACIÓN DE ACCIONES")
print("-" * 100)
summary = action_space_def['action_summary']
summary_data = [
    {'Categoría': 'Acciones de movimiento', 'Acciones': str(summary['movement_actions'])},
    {'Categoría': 'Acciones de interacción', 'Acciones': str(summary['object_interaction_actions'])},
    {'Categoría': 'Siempre aplicables', 'Acciones': str(summary['always_applicable'])},
    {'Categoría': 'Condicionalmente aplicables', 'Acciones': str(summary['conditionally_applicable'])},
]
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n\nTABLA 2: DETALLES DE CADA ACCIÓN")
print("-" * 100)
actions_data = []
for action_name, action_details in action_space_def['actions'].items():
    applicability = action_details['applicability']
    actions_data.append({
        'Acción': action_name,
        'Nombre': action_details['name'],
        'Tipo': action_details['type'],
        'Condición': applicability.get('condition', 'N/A'),
        'Válida cuando': applicability.get('valid_when', 'N/A')[:60] + '...' if len(applicability.get('valid_when', 'N/A')) > 60 else applicability.get('valid_when', 'N/A'),
    })
df_actions = pd.DataFrame(actions_data)
print(df_actions.to_string(index=False))

print("\n\nTABLA 3: RECOMPENSAS POR ACCIÓN")
print("-" * 100)
rewards_data = []
for action_name, action_details in action_space_def['actions'].items():
    applicability = action_details['applicability']
    rewards_data.append({
        'Acción': action_name,
        'Recompensa (válida)': applicability.get('reward_if_valid', 'N/A'),
        'Recompensa (inválida)': applicability.get('reward_if_invalid', 'N/A'),
    })
df_rewards = pd.DataFrame(rewards_data)
print(df_rewards.to_string(index=False))

print("\n\nTABLA 4: UBICACIONES CLAVE Y SUS INTERACCIONES")
print("-" * 100)
transitions = action_space_def['state_transitions']
transitions_data = []
for location_type, location_info in transitions.items():
    pos = location_info.get('start', location_info.get('position', 'N/A'))
    if isinstance(pos, tuple) and len(pos) == 2:
        pos_text = f"(R={pos[0]}, C={pos[1]})"
    else:
        pos_text = str(pos)
    transitions_data.append({
        'Ubicación': location_type,
        'Posición': pos_text,
        'Interacción': location_info.get('interaction', 'N/A'),
    })
df_transitions = pd.DataFrame(transitions_data)
print(df_transitions.to_string(index=False))

# ============================================================================
# GUARDAR CELDA 5: DEFINICIÓN DE RECOMPENSAS
# ============================================================================
output_buffer = io.StringIO()
sys.stdout = output_buffer

reward_def = env.get_reward_function_definition()

print("=" * 120)
print("DEFINICIÓN DE LA FUNCIÓN DE RECOMPENSA - DoorKeyBall Environment")
print("=" * 120)

print(f"\nDescripción: {reward_def['description']}")
print(f"Tipos totales de recompensas: {reward_def['total_reward_types']}\n")

print("TABLA 1: RESUMEN DE TIPOS DE RECOMPENSAS")
print("-" * 120)
rewards_summary = []
for reward_type, reward_info in reward_def['reward_values'].items():
    rewards_summary.append({
        'Tipo': reward_type,
        'Valor': reward_info['value'],
        'Descripción': reward_info['description'],
        'Cuándo aplica': reward_info['when_applicable'],
        'Motivación': reward_info['motivation']
    })
df_rewards_summary = pd.DataFrame(rewards_summary)
print(df_rewards_summary.to_string(index=False))

print("\n\nTABLA 2: ESTRUCTURA DE RECOMPENSAS")
print("-" * 120)
reward_struct = reward_def['reward_structure']
structure_data = [
    {'Aspecto': 'Tipo de estructura', 'Valor': reward_struct['type']},
    {'Aspecto': 'Escala de valores', 'Valor': reward_struct['scale']},
    {'Aspecto': 'Recompensas negativas (penalizaciones)', 'Valor': str(reward_struct['negative_rewards'])},
    {'Aspecto': 'Recompensas positivas (incentivos)', 'Valor': str(reward_struct['positive_rewards'])},
]
df_structure = pd.DataFrame(structure_data)
print(df_structure.to_string(index=False))

print("\n\nTABLA 3: SITUACIONES (PARES ESTADO-ACCIÓN) QUE GENERAN RECOMPENSA")
print("-" * 120)
situations_data = []
for reward_type, reward_info in reward_def['reward_values'].items():
    situations_text = " | ".join(reward_info['situations'])
    situations_data.append({
        'Recompensa': reward_type,
        'Valor': reward_info['value'],
        'Situaciones': situations_text
    })
df_situations = pd.DataFrame(situations_data)
print(df_situations.to_string(index=False))

print("\n\nTABLA 4: EJEMPLO DE RUTA ÓPTIMA (CAMINO ÓPTIMO)")
print("-" * 120)
example = reward_def['cumulative_reward_example']
print(f"Escenario: {example['scenario']}\n")
route_data = []
for i, step in enumerate(example['path'], 1):
    route_data.append({'Paso': i, 'Transición': step})
df_route = pd.DataFrame(route_data)
print(df_route.to_string(index=False))
print(f"\nRecompensa total óptima: {example['total_optimal']}")

print("\n\nTABLA 5: ESTADO - ACCIÓN - RECOMPENSA")
print("-" * 120)
state_action_data = reward_def['state_action_reward_table']
df_state_action = pd.DataFrame(state_action_data)
print(df_state_action.to_string(index=False))

print("\n" + "=" * 120)
print("RESUMEN FINAL: 8 tipos de recompensas definiendo todas las parejas (estado, acción)")
print("=" * 120)

# Guardar en archivo
sys.stdout = old_stdout
output_text_cell5 = output_buffer.getvalue()
with open(os.path.join(output_dir, "celda_5_funcion_recompensa.txt"), "w", encoding="utf-8") as f:
    f.write(output_text_cell5)
print(f"✓ Salida de celda 3 guardada en: {output_dir}/celda_3_espacio_de_estados.txt")
print(f"✓ Salida de celda 4 guardada en: {output_dir}/celda_4_espacio_de_acciones.txt")
print(f"✓ Salida de celda 5 guardada en: {output_dir}/celda_5_funcion_recompensa.txt")

✓ Salida de celda 3 guardada en: outputs_definiciones/celda_3_espacio_de_estados.txt
✓ Salida de celda 3 guardada en: outputs_definiciones/celda_3_espacio_de_estados.txt
✓ Salida de celda 4 guardada en: outputs_definiciones/celda_4_espacio_de_acciones.txt
✓ Salida de celda 5 guardada en: outputs_definiciones/celda_5_funcion_recompensa.txt


In [115]:
# Demostración: max_steps=50 por defecto
print("=" * 80)
print("DEMOSTRACIÓN: max_steps=50 por defecto")
print("=" * 80)

# Crear entorno sin especificar max_steps (usa 50 por defecto)
env_default = DoorKeyBallEnvironment()
print(f"\nEntorno creado sin especificar max_steps:")
print(f"  Parámetro max_steps del entorno: {env_default.max_steps}")

# Simular episodio corto (solo movimientos en LR)
state = env_default.reset()
step_count = 0
total_reward = 0
print(f"\nSimulación de episodio corto (movimientos en LR): ")
print(f"  Estado inicial: {state}, step_count={env_default._step_count}")

# Solo movimientos LEFT/RIGHT en LR (C<5)
actions_sequence = ["RIGHT"] * 4 + ["LEFT"] * 3
for action in actions_sequence:
    state, reward, done = env_default.step(state, action)
    step_count += 1
    total_reward += reward
    if step_count <= 100 or done:
        print(f"  Paso {step_count}: Acción={action}, Recompensa={reward}, Done={done}, Estado={state}")
    if done:
        break

print(f"\nTotal de pasos ejecutados: {step_count}")
print(f"Recompensa acumulada: {total_reward:.2f}")

# Crear entorno con max_steps=10 para ver término por límite de pasos
print("\n" + "=" * 80)
print("Comparación: Entorno con max_steps=10")
print("=" * 80)

env_short = DoorKeyBallEnvironment(max_steps=10)
print(f"\nEntorno creado con max_steps=10")
print(f"  Parámetro max_steps del entorno: {env_short.max_steps}")

state = env_short.reset()
print(f"\nSimulación de episodio (movimientos en RR para ver 10 pasos):")
print(f"  Estado inicial: {state}")

# Intentar 15 movimientos (pero deberían terminar en 10)
total_reward_short = 0
for i in range(15):
    state, reward, done = env_short.step(state, "UP")
    total_reward_short += reward
    if i < 100 or done:
        print(f"  Paso {i+1}: Recompensa={reward}, Done={done}, _step_count={env_short._step_count}")
    if done:
        print(f"  ... (episodio terminó en paso {env_short._step_count})")
        break

print(f"\nEpisodio terminado porque alcanzó max_steps={env_short.max_steps}")
print(f"Recompensa acumulada: {total_reward_short:.2f}")


DEMOSTRACIÓN: max_steps=50 por defecto

Entorno creado sin especificar max_steps:
  Parámetro max_steps del entorno: 50

Simulación de episodio corto (movimientos en LR): 
  Estado inicial: (3, 1, 0, 0, 0), step_count=0
  Paso 1: Acción=RIGHT, Recompensa=-0.02, Done=False, Estado=(3, 2, 0, 0, 0)
  Paso 2: Acción=RIGHT, Recompensa=-0.02, Done=False, Estado=(3, 3, 0, 0, 0)
  Paso 3: Acción=RIGHT, Recompensa=-0.02, Done=False, Estado=(3, 4, 0, 0, 0)
  Paso 4: Acción=RIGHT, Recompensa=-0.1, Done=False, Estado=(3, 4, 0, 0, 0)
  Paso 5: Acción=LEFT, Recompensa=-0.02, Done=False, Estado=(3, 3, 0, 0, 0)
  Paso 6: Acción=LEFT, Recompensa=-0.02, Done=False, Estado=(3, 2, 0, 0, 0)
  Paso 7: Acción=LEFT, Recompensa=-0.02, Done=False, Estado=(3, 1, 0, 0, 0)

Total de pasos ejecutados: 7
Recompensa acumulada: -0.22

Comparación: Entorno con max_steps=10

Entorno creado con max_steps=10
  Parámetro max_steps del entorno: 10

Simulación de episodio (movimientos en RR para ver 10 pasos):
  Estado inici